# Baseline — Rosetta Embedding Alignment

**Competition:** two embedding spaces, A (384-d) and B (256-d), encode the same kind of
items but come from *different models*. You get **32 anchor pairs** (the same item in
both spaces). For each of 1200 `query_A` vectors, find the matching vector among 1500
`gallery_B` candidates.

- **Task:** cross-space retrieval — submit the gallery index per query
- **Metric:** retrieval accuracy
- **Kaggle link:** _TODO: add link_

**Approach:** learn a **linear map** from space A to space B on the 32 anchors
(ridge-regularized least squares), then nearest-neighbor by cosine similarity.
`practice_1/` and `practice_2/` include ground truth to validate the method.

In [1]:
import numpy as np
import pandas as pd

DATA_DIR = "data"

def load(d):
    return {n: np.load(f"{d}/{n}.npy") for n in
            ["anchor_A", "anchor_B", "query_A", "gallery_B"]}

def norm(x):
    return x / np.linalg.norm(x, axis=1, keepdims=True)

def fit_linear_map(A, B, lam=1e-2):
    # W minimizing ||A W - B||^2 + lam ||W||^2
    d = A.shape[1]
    return np.linalg.solve(A.T @ A + lam * np.eye(d), A.T @ B)

def retrieve(data, lam=1e-2):
    W = fit_linear_map(data["anchor_A"], data["anchor_B"], lam)
    q = norm(data["query_A"] @ W)
    g = norm(data["gallery_B"])
    return np.argmax(q @ g.T, axis=1)

In [2]:
# Validate on the practice sets (they ship with truth.csv)
for p in ["practice_1", "practice_2"]:
    d = load(f"{DATA_DIR}/{p}")
    truth = pd.read_csv(f"{DATA_DIR}/{p}/truth.csv")
    pred = retrieve(d)
    acc = (pred == truth.iloc[:, -1].values).mean()
    print(f"{p}: retrieval accuracy = {acc:.4f}")

practice_1: retrieval accuracy = 0.1825


practice_2: retrieval accuracy = 0.2208


In [3]:
# Predict the real test queries
data = load(DATA_DIR)
pred = retrieve(data)
sub = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
sub["gallery_id"] = pred
sub.to_csv("submission.csv", index=False)
sub.head()

,id,gallery_id
0,q_0000,1444
1,q_0001,1215
2,q_0002,1300
3,q_0003,1036
4,q_0004,1205


## Ideas to improve

- **Orthogonal Procrustes** (SVD of `A.T @ B`) instead of ridge — rotation-only maps
  often generalize better from few anchors.
- Center + scale both spaces before mapping; tune the ridge `lam` on the practice sets.
- **CSLS** re-ranking instead of plain cosine (fixes hubness in cross-space retrieval).
- Iterative self-learning: take confident matches as new anchors and refit.
